In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import expr, col

import ConnectionConfig as cc
cc.setupEnvironment()

In [3]:
spark = cc.startLocalCluster("STATION_DIM",4)
spark.getActiveSession()

In [4]:
# load initial station data
df_stations = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stations") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "stationid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()


df_stations.show()


+---------+--------+---------+------------+--------------------+-------+-------+----------+-----------------+--------------------+-------+------+
|stationid|objectid|stationnr|        type|              street| number|zipcode|  district|         gpscoord|      additionalinfo|labelid|cityid|
+---------+--------+---------+------------+--------------------+-------+-------+----------+-----------------+--------------------+-------+------+
|        1|   33202|      026|DUBBELZIJDIG|         Meir (2000)|     84|   2000| ANTWERPEN|(51.2182,4.41241)|                    |   NULL|  NULL|
|        2|   33203|      019| ENKELZIJDIG|          ONTBREKEND|     12|   2000| ANTWERPEN| (51.219,4.40405)|                    |   NULL|  NULL|
|        3|   33204|      020| ENKELZIJDIG|Groenkerkhofstraa...|      2|   2000| ANTWERPEN|(51.2187,4.40066)| thv Nationalestraat|   NULL|  NULL|
|        4|   33205|      035| ENKELZIJDIG|Cockerillkaai (2000)|       |   2000| ANTWERPEN|(51.2104,4.38772)|               

In [6]:
from pyspark.sql.functions import expr

df_station_dim = df_stations.select(
    expr("uuid()").alias("station_sk"),
    col("stationid"),
    col("stationnr"),
    col("street"),
    col("number"),
    col("zipcode"),
    col("district"),
    col("gpscoord")
)

df_station_dim.show()


+--------------------+---------+---------+--------------------+-------+-------+----------+-----------------+
|          station_sk|stationid|stationnr|              street| number|zipcode|  district|         gpscoord|
+--------------------+---------+---------+--------------------+-------+-------+----------+-----------------+
|01c13fd4-dce7-45a...|        1|      026|         Meir (2000)|     84|   2000| ANTWERPEN|(51.2182,4.41241)|
|aa529b3e-557f-455...|        2|      019|          ONTBREKEND|     12|   2000| ANTWERPEN| (51.219,4.40405)|
|37716947-beb6-4b1...|        3|      020|Groenkerkhofstraa...|      2|   2000| ANTWERPEN|(51.2187,4.40066)|
|0603dfa1-d68d-499...|        4|      035|Cockerillkaai (2000)|       |   2000| ANTWERPEN|(51.2104,4.38772)|
|cb91a863-d02c-406...|        5|      094|        PALEISSTRAAT|    147|   2018| ANTWERPEN|(51.2047,4.39625)|
|bc04c945-d16a-4e9...|        6|      012|       Singel (2018)|       |   2018| ANTWERPEN|(51.1994,4.39013)|
|27c3c83f-3b07-42e.

In [7]:
spark.sql("DROP TABLE IF EXISTS stationdim")

df_station_dim.write.format("delta").mode("overwrite").saveAsTable("stationdim")


In [9]:
# export to database
df_station_dim.write \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stationdim") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("batchsize", 1000) \
    .mode("overwrite") \
    .save()


In [10]:
df_verify = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stationdim") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_verify.show()


+--------------------+---------+---------+--------------------+------+-------+----------+-----------------+
|          station_sk|stationid|stationnr|              street|number|zipcode|  district|         gpscoord|
+--------------------+---------+---------+--------------------+------+-------+----------+-----------------+
|01c13fd4-dce7-45a...|        1|      026|         Meir (2000)|    84|   2000| ANTWERPEN|(51.2182,4.41241)|
|aa529b3e-557f-455...|        2|      019|          ONTBREKEND|    12|   2000| ANTWERPEN| (51.219,4.40405)|
|37716947-beb6-4b1...|        3|      020|Groenkerkhofstraa...|     2|   2000| ANTWERPEN|(51.2187,4.40066)|
|e217535c-9d72-42f...|      250|      297|Jan De Voslei (2020)|      |   2020| ANTWERPEN| (51.1907,4.3889)|
|0603dfa1-d68d-499...|        4|      035|Cockerillkaai (2000)|      |   2000| ANTWERPEN|(51.2104,4.38772)|
|cb91a863-d02c-406...|        5|      094|        PALEISSTRAAT|   147|   2018| ANTWERPEN|(51.2047,4.39625)|
|bf26067d-83cb-4b9...|      

In [11]:
spark.stop()


WHAT GOES DOWN IS FOR TESTING REASONS, RUN IT ONLY FOR IT(ITS ALREADY BEEN TESTED BY RUFINA, SO YOU DONT NEED TO DO IT AGAIN;)


In [14]:
# Load data again to test the initial load
df_stations_test = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stations") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

# Display the data to confirm the load
df_stations_test.show(truncate=False)

Py4JJavaError: An error occurred while calling o192.showString.
: java.lang.IllegalStateException: SparkContext has been shutdown
	at org.apache.spark.SparkContext.runJob(SparkContext.scala:2385)
	at org.apache.spark.SparkContext.runJob(SparkContext.scala:2414)
	at org.apache.spark.SparkContext.runJob(SparkContext.scala:2433)
	at org.apache.spark.sql.execution.SparkPlan.executeTake(SparkPlan.scala:530)
	at org.apache.spark.sql.execution.SparkPlan.executeTake(SparkPlan.scala:483)
	at org.apache.spark.sql.execution.CollectLimitExec.executeCollect(limit.scala:61)
	at org.apache.spark.sql.Dataset.collectFromPlan(Dataset.scala:4334)
	at org.apache.spark.sql.Dataset.$anonfun$head$1(Dataset.scala:3316)
	at org.apache.spark.sql.Dataset.$anonfun$withAction$2(Dataset.scala:4324)
	at org.apache.spark.sql.execution.QueryExecution$.withInternalError(QueryExecution.scala:546)
	at org.apache.spark.sql.Dataset.$anonfun$withAction$1(Dataset.scala:4322)
	at org.apache.spark.sql.execution.SQL

In [18]:
# Check data in the Delta table
spark.sql("SELECT * FROM stationdim").show(truncate=False)

# Check the table schema
spark.sql("DESCRIBE TABLE stationdim").show(truncate=False)


+------------------------------------+---------+---------+-----------------------------+------+-------+---------+-----------------+
|station_sk                          |stationid|stationnr|street                       |number|zipcode|district |gpscoord         |
+------------------------------------+---------+---------+-----------------------------+------+-------+---------+-----------------+
|84b54607-d7b1-46dc-9997-3738f3605332|250      |297      |Jan De Voslei (2020)         |      |2020   |ANTWERPEN|(51.1907,4.3889) |
|cb7ee1df-74a4-4921-8803-f881ed961003|251      |298      |Valkstraat (2610)            |      |2610   |WILRIJK  |(51.1719,4.38248)|
|ede3d861-7f43-485d-a94e-9e9af960bcc6|252      |299      |Camille Huysmanslaan (2020)  |      |2020   |ANTWERPEN|(51.192,4.39806) |
|bdabf0ee-7178-4b7d-9a54-c845385af7fa|253      |300      |Vogelzanglaan (2020)         |      |2020   |ANTWERPEN|(51.1894,4.39731)|
|8c791be5-1f2e-48e2-852b-dc49e40b5255|254      |301      |Luchthavenlei (210

In [19]:
# load data from PostgreSQL to verify export
df_stationdim_pg = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stationdim") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_stationdim_pg.show(truncate=False)


+------------------------------------+---------+---------+-----------------------------+------+-------+---------+-----------------+
|station_sk                          |stationid|stationnr|street                       |number|zipcode|district |gpscoord         |
+------------------------------------+---------+---------+-----------------------------+------+-------+---------+-----------------+
|84b54607-d7b1-46dc-9997-3738f3605332|250      |297      |Jan De Voslei (2020)         |      |2020   |ANTWERPEN|(51.1907,4.3889) |
|cb7ee1df-74a4-4921-8803-f881ed961003|251      |298      |Valkstraat (2610)            |      |2610   |WILRIJK  |(51.1719,4.38248)|
|ede3d861-7f43-485d-a94e-9e9af960bcc6|252      |299      |Camille Huysmanslaan (2020)  |      |2020   |ANTWERPEN|(51.192,4.39806) |
|bdabf0ee-7178-4b7d-9a54-c845385af7fa|253      |300      |Vogelzanglaan (2020)         |      |2020   |ANTWERPEN|(51.1894,4.39731)|
|8c791be5-1f2e-48e2-852b-dc49e40b5255|254      |301      |Luchthavenlei (210

In [20]:
#after insertion the following line in the sql console:
# UPDATE stations SET street = 'New Street Name' WHERE stationid = 1;





# load the updated stations data
df_stations_updated = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stations") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_stations_updated.show()

# Transform and prepare the data again
df_station_dim_updated = df_stations_updated.select(
    expr("uuid()").alias("station_sk"),
    col("stationid"),
    col("stationnr"),
    col("street"),
    col("number"),
    col("zipcode"),
    col("district"),
    col("gpscoord")
)

df_station_dim_updated.show()


+---------+--------+---------+------------+--------------------+-------+-------+----------+-----------------+--------------------+-------+------+
|stationid|objectid|stationnr|        type|              street| number|zipcode|  district|         gpscoord|      additionalinfo|labelid|cityid|
+---------+--------+---------+------------+--------------------+-------+-------+----------+-----------------+--------------------+-------+------+
|        2|   33203|      019| ENKELZIJDIG|          ONTBREKEND|     12|   2000| ANTWERPEN| (51.219,4.40405)|                    |   NULL|  NULL|
|        3|   33204|      020| ENKELZIJDIG|Groenkerkhofstraa...|      2|   2000| ANTWERPEN|(51.2187,4.40066)| thv Nationalestraat|   NULL|  NULL|
|        4|   33205|      035| ENKELZIJDIG|Cockerillkaai (2000)|       |   2000| ANTWERPEN|(51.2104,4.38772)|                    |   NULL|  NULL|
|        5|   33206|      094| ENKELZIJDIG|        PALEISSTRAAT|    147|   2018| ANTWERPEN|(51.2047,4.39625)|               

In [22]:
# Check the Delta table again to confirm no changes
spark.sql("SELECT * FROM stationdim WHERE stationid=1").show(truncate=False)


+------------------------------------+---------+---------+-----------+------+-------+---------+-----------------+
|station_sk                          |stationid|stationnr|street     |number|zipcode|district |gpscoord         |
+------------------------------------+---------+---------+-----------+------+-------+---------+-----------------+
|1f7dbcf3-3232-40da-87a2-44313faf82d4|1        |026      |Meir (2000)|84    |2000   |ANTWERPEN|(51.2182,4.41241)|
+------------------------------------+---------+---------+-----------+------+-------+---------+-----------------+



It works well, after running an update command in the console:UPDATE stations SET street = 'New Street Name' WHERE stationid = 1;

The Delta table still shows the original data, even if the source PostgreSQL table has changed.
This confirms that Type 0 is respected.